# 06. Train Generalized Matrix Factorization (GMF)

Trains the GMF network using Adam optimizer and evaluates it on test data.

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import json

np.random.seed(42)
tf.random.set_seed(42)

processed_dir = os.path.join("..", "..", "datasets", "processed")
test_df = pd.read_csv(os.path.join(processed_dir, "test.csv"))
full_df = pd.read_csv(os.path.join(processed_dir, "recommendation_processed.csv"))

train_data = np.load(os.path.join(processed_dir, "train_neg.npz"))
val_data = np.load(os.path.join(processed_dir, "val_neg.npz"))

train_u, train_i, train_y = train_data["u"], train_data["i"], train_data["y"]
val_u, val_i, val_y = val_data["u"], val_data["i"], val_data["y"]

num_users = int(full_df["user_idx"].max() + 1)
num_items = int(full_df["item_idx"].max() + 1)
all_items = set(range(num_items))

def build_gmf(num_users, num_items, latent_dim=16):
    user_input = tf.keras.layers.Input(shape=(1,), dtype="int32", name="user_input")
    item_input = tf.keras.layers.Input(shape=(1,), dtype="int32", name="item_input")
    
    user_embed = tf.keras.layers.Embedding(num_users, latent_dim, name="user_embed")(user_input)
    item_embed = tf.keras.layers.Embedding(num_items, latent_dim, name="item_embed")(item_input)
    
    user_flat = tf.keras.layers.Flatten()(user_embed)
    item_flat = tf.keras.layers.Flatten()(item_embed)
    
    prod = tf.keras.layers.Multiply()([user_flat, item_flat])
    output = tf.keras.layers.Dense(1, activation="sigmoid", name="output")(prod)
    
    model = tf.keras.models.Model(inputs=[user_input, item_input], outputs=output)
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

gmf = build_gmf(num_users, num_items)
history = gmf.fit(
    [train_u, train_i], train_y, 
    validation_data=([val_u, val_i], val_y),
    epochs=3, 
    batch_size=256,
    verbose=1
)

# Save model weights and full model
artifacts_dir = os.path.join("..", "..", "artifacts", "recommendation")
saved_models_dir = os.path.join("..", "..", "saved_models", "recommendation")
os.makedirs(artifacts_dir, exist_ok=True)
os.makedirs(saved_models_dir, exist_ok=True)
gmf.save(os.path.join(artifacts_dir, "gmf_model.keras"))
gmf.save(os.path.join(saved_models_dir, "gmf_model.keras"))

# Batched evaluation to prevent loop overhead
def evaluate_keras_model_batched(model, test_df, all_items, topK=10):
    user_list = []
    item_list = []
    for idx, row in test_df.iterrows():
        u = int(row["user_idx"])
        pos_i = int(row["item_idx"])
        negs = list(all_items - {pos_i})
        if len(negs) > 99:
            sampled_negs = list(np.random.choice(negs, 99, replace=False))
        else:
            sampled_negs = negs
        candidates = [pos_i] + sampled_negs
        user_list.append([u] * len(candidates))
        item_list.append(candidates)
        
    user_arr = np.array(user_list).flatten()
    item_arr = np.array(item_list).flatten()
    
    preds = model.predict([user_arr, item_arr], batch_size=4096, verbose=0).flatten()
    preds = preds.reshape(len(test_df), -1)
    
    hr_list, ndcg_list = [], []
    for idx, (row, pred) in enumerate(zip(test_df.itertuples(), preds)):
        pos_i = int(row.item_idx)
        candidates = item_list[idx]
        
        top_indices = pred.argsort()[::-1][:topK]
        top_items = [candidates[i] for i in top_indices]
        
        if pos_i in top_items:
            hr_list.append(1.0)
            rank = top_items.index(pos_i)
            ndcg_list.append(1.0 / np.log2(rank + 2))
        else:
            hr_list.append(0.0)
            ndcg_list.append(0.0)
            
    return np.mean(hr_list), np.mean(ndcg_list)

hr_10, ndcg_10 = evaluate_keras_model_batched(gmf, test_df, all_items)
print(f"GMF HR@10: {hr_10:.4f}, NDCG@10: {ndcg_10:.4f}")

best_epoch = int(np.argmin(history.history['val_loss']) + 1)
metrics = {
    "gmf_hr_10": hr_10,
    "gmf_ndcg_10": ndcg_10,
    "gmf_best_epoch": best_epoch
}
with open(os.path.join("..", "..", "outputs", "recommendation", "gmf_metrics.json"), "w") as f:
    json.dump(metrics, f)

print("GMF model training and evaluation complete.")